# Repeated-Seed Aggregation and Final Evaluation Preparation

This notebook documents the aggregation of three completed stacking-and-calibration seeds. It attaches locked-test silver labels only after label-blind prediction artifacts are complete, verifies consistency across seeds, and prepares aggregate evaluation outputs for subsequent human-label, statistical, and triage analyses.


## Release Boundary

This restricted code-inspection copy retains the analytical logic but does not include Banjarbaru review data, row-level predictions, checkpoints, fold models, or private run artifacts. The restricted path placeholder documents the authorized environment and is not an executable public-data contract.


### Configure the repeated-seed aggregation contract

Imports evaluation utilities, defines the restricted project-root placeholder, specifies the stacking input, aggregation output, locked-test label source, planned seeds, model order, protocol versions, and evaluation metrics.


In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score


PROJECT_ROOT = Path('RESTRICTED_PROJECT_ROOT')
ROUND1_DIR = PROJECT_ROOT / 'journal_round1'
R1_07_ROOT = ROUND1_DIR / 'artifacts' / 'r1_07_revised_stacking_calibration_v1'
OUTPUT_ROOT = ROUND1_DIR / 'artifacts' / 'r1_08_repeated_seed_evaluation_v1'
LOCKED_TEST_SOURCE_PATH = ROUND1_DIR / 'data' / 'final_test_locked' / 'test_locked.csv'

PLANNED_SEEDS = [42, 52, 62]
LABEL_NAMES = ['negatif', 'netral', 'positif']
LABEL_TO_ID = {label: index for index, label in enumerate(LABEL_NAMES)}
N_CLASSES = len(LABEL_NAMES)
LABEL_COLUMN = 'sentiment_label'
MODEL_ORDER = [
    ('svm', 'SVM'),
    ('indobert', 'IndoBERT'),
    ('wa_svm_bert', 'WA (SVM+BERT)'),
    ('xgboost', 'XGBoost'),
    ('calibrated_xgboost', 'Calibrated XGBoost'),
]
R1_07_PROTOCOL_VERSION = 'r1_07_protocol_v1_oof_stacking_validation_platt'
R1_06_PROTOCOL_VERSION = 'r1_06_protocol_v2_outer_train_refit'
PROTOCOL_VERSION = 'r1_08_protocol_v1_repeated_seed_silver_evaluation'
METRIC_COLUMNS = ['accuracy', 'macro_f1', 'ece', 'brier_score']
VALIDATION_METRIC_REPRODUCTION_ATOL = 2e-7

print(f'Restricted project root placeholder: {PROJECT_ROOT}')
print(f'Stacking and calibration input: {R1_07_ROOT}')
print(f'Repeated-seed evaluation output: {OUTPUT_ROOT}')
print(f'Locked-test silver-label source: {LOCKED_TEST_SOURCE_PATH}')
print(f'Fixed seeds: {PLANNED_SEEDS}')


### Define evaluation, checksum, and persistence utilities

Defines atomic writers, probability and row-order checks, entropy, ECE, Brier score, metric summaries, and manifest helpers for aggregation.


In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_json_hash(payload: dict) -> str:
    raw = json.dumps(payload, sort_keys=True, separators=(',', ':'), ensure_ascii=True).encode('utf-8')
    return hashlib.sha256(raw).hexdigest()


def atomic_write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'.{path.name}.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=True, default=float), encoding='utf-8')
    temporary.replace(path)


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'.{path.name}.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def probability_columns(model_key: str) -> list[str]:
    return [f'{model_key}_prob_{label}' for label in LABEL_NAMES]


def validate_probability_matrix(probabilities: np.ndarray, name: str) -> None:
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.ndim != 2 or probabilities.shape[1] != N_CLASSES:
        raise ValueError(f'{name}: bentuk probability tidak valid: {probabilities.shape}')
    if not np.isfinite(probabilities).all():
        raise ValueError(f'{name}: terdapat NaN atau infinity.')
    if np.any(probabilities < -1e-10) or np.any(probabilities > 1 + 1e-10):
        raise ValueError(f'{name}: probability di luar [0, 1].')
    maximum_error = float(np.max(np.abs(probabilities.sum(axis=1) - 1.0)))
    if maximum_error > 1e-5:
        raise ValueError(f'{name}: jumlah probability per row tidak satu. Error={maximum_error:.3e}')


def entropy(probabilities: np.ndarray) -> np.ndarray:
    safe = np.clip(probabilities, 1e-12, 1.0)
    return -(safe * np.log(safe)).sum(axis=1)


def compute_ece(probabilities: np.ndarray, y_true: np.ndarray, n_bins: int = 10) -> float:
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    class_scores = []
    for class_index in range(N_CLASSES):
        probability = probabilities[:, class_index]
        actual = (y_true == class_index).astype(float)
        score = 0.0
        for left, right in zip(edges[:-1], edges[1:]):
            mask = (probability >= left) & ((probability <= right) if right == 1.0 else (probability < right))
            if mask.any():
                score += float(mask.mean() * abs(actual[mask].mean() - probability[mask].mean()))
        class_scores.append(score)
    return float(np.mean(class_scores))


def compute_brier(probabilities: np.ndarray, y_true: np.ndarray) -> float:
    return float(np.mean([
        brier_score_loss((y_true == class_index).astype(float), probabilities[:, class_index])
        for class_index in range(N_CLASSES)
    ]))


def metric_summary(y_true: np.ndarray, probabilities: np.ndarray) -> dict:
    validate_probability_matrix(probabilities, 'metric probabilities')
    prediction = probabilities.argmax(axis=1)
    return {
        'n': int(len(y_true)),
        'accuracy': float(accuracy_score(y_true, prediction)),
        'macro_f1': float(f1_score(y_true, prediction, average='macro', zero_division=0)),
        'ece': compute_ece(probabilities, y_true),
        'brier_score': compute_brier(probabilities, y_true),
    }


def assert_exact_row_ids(frame: pd.DataFrame, expected_row_ids: np.ndarray, name: str) -> None:
    if 'row_id' not in frame.columns:
        raise ValueError(f'{name}: kolom row_id tidak ditemukan.')
    actual = frame['row_id'].to_numpy(dtype=np.int64)
    if not np.array_equal(actual, expected_row_ids):
        raise ValueError(f'{name}: row_id atau urutan row tidak sama dengan locked-test contract.')


def validate_model_outputs(frame: pd.DataFrame, name: str, require_label: bool) -> None:
    if require_label and 'y_silver' not in frame.columns:
        raise ValueError(f'{name}: y_silver wajib tersedia.')
    if not require_label and ('y_silver' in frame.columns or any('label' in column for column in frame.columns)):
        raise ValueError(f'{name}: artifact label-blind memuat label.')
    if require_label:
        labels = frame['y_silver'].to_numpy(dtype=np.int64)
        if not np.isin(labels, np.arange(N_CLASSES)).all():
            raise ValueError(f'{name}: y_silver di luar kelas yang diizinkan.')
    for model_key, _ in MODEL_ORDER:
        probabilities = frame[probability_columns(model_key)].to_numpy(dtype=np.float64)
        validate_probability_matrix(probabilities, f'{name}/{model_key}')
        predicted = frame[f'{model_key}_prediction'].to_numpy(dtype=np.int64)
        if not np.array_equal(predicted, probabilities.argmax(axis=1)):
            raise ValueError(f'{name}/{model_key}: hard prediction tidak cocok dengan argmax probability.')
        if not np.allclose(frame[f'{model_key}_confidence'].to_numpy(dtype=np.float64), probabilities.max(axis=1), atol=1e-7):
            raise ValueError(f'{name}/{model_key}: confidence tidak cocok dengan probability.')
        if not np.allclose(frame[f'{model_key}_entropy'].to_numpy(dtype=np.float64), entropy(probabilities), atol=1e-7):
            raise ValueError(f'{name}/{model_key}: entropy tidak cocok dengan probability.')


### Attach locked-test silver labels after blind prediction completion

Reads only the locked-test silver-label column after upstream label-blind predictions are complete, maps it to numeric labels, and validates the fixed locked-test row contract.


In [ ]:
if not LOCKED_TEST_SOURCE_PATH.is_file():
    raise FileNotFoundError(f'File locked test tidak ditemukan: {LOCKED_TEST_SOURCE_PATH}')

locked_test_source = pd.read_csv(LOCKED_TEST_SOURCE_PATH, usecols=[LABEL_COLUMN])
locked_test_labels_text = locked_test_source[LABEL_COLUMN].fillna('').astype(str).str.strip().str.lower()
unknown_labels = sorted(set(locked_test_labels_text).difference(LABEL_TO_ID))
if unknown_labels:
    raise ValueError(f'Locked test memiliki silver label tidak dikenal: {unknown_labels}')

LOCKED_ROW_IDS = np.arange(len(locked_test_source), dtype=np.int64)
LOCKED_Y_SILVER = locked_test_labels_text.map(LABEL_TO_ID).to_numpy(dtype=np.int64)
LOCKED_TEST_SOURCE_SHA256 = sha256_file(LOCKED_TEST_SOURCE_PATH)

print(f'Locked test loaded for final silver-label evaluation: {len(LOCKED_Y_SILVER):,} rows')
print('Silver-label class counts:')
display(pd.Series(locked_test_labels_text).value_counts().reindex(LABEL_NAMES, fill_value=0).rename_axis('label').reset_index(name='n'))


### Load and validate stacking-and-calibration seed outputs

Checks the five final-model predictions, protocol compatibility, model order, checksums, label-blind locked-test constraint, validation metric reproducibility, and upstream base-model manifest linkage for each seed.


In [ ]:
R1_07_REQUIRED_OUTPUTS = [
    'models/xgboost_12_feature.joblib',
    'models/calibrated_xgboost_platt_full_validation.joblib',
    'oof_model_predictions_fit_only.csv',
    'validation_model_predictions.csv',
    'locked_test_model_predictions_blind.csv',
    'validation_metrics.csv',
    'calibration_crossfit_validation_predictions.csv',
    'calibration_crossfit_summary.csv',
    'weighted_average_summary.json',
    'xgboost_feature_importance_gain.csv',
    'schema.json',
]


def load_and_validate_r1_07_seed(seed: int) -> dict:
    seed_dir = R1_07_ROOT / f'seed_{seed}'
    manifest_path = seed_dir / 'run_manifest.json'
    state_path = seed_dir / 'run_state.json'
    required_paths = [seed_dir / relative_path for relative_path in [*R1_07_REQUIRED_OUTPUTS, 'run_manifest.json', 'run_state.json']]
    missing = [str(path.relative_to(seed_dir)) for path in required_paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f'Stacking seed {seed} belum lengkap: {missing}')

    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    state = json.loads(state_path.read_text(encoding='utf-8'))
    protocol = manifest.get('protocol', {})
    if manifest.get('seed') != seed or state.get('seed') != seed or state.get('status') != 'seed_completed':
        raise RuntimeError(f'Stacking seed {seed} tidak berstatus seed_completed.')
    if protocol.get('protocol_version') != R1_07_PROTOCOL_VERSION:
        raise RuntimeError(f'Stacking seed {seed} memakai protocol yang tidak didukung.')
    if protocol.get('source_r1_06_protocol_version') != R1_06_PROTOCOL_VERSION:
        raise RuntimeError(f'Stacking seed {seed} bukan keluaran base-model outer-train refit.')
    if [tuple(item) for item in protocol.get('model_order', [])] != MODEL_ORDER:
        raise RuntimeError(f'Stacking seed {seed}: urutan lima model final berubah.')
    if manifest.get('locked_test_metrics_computed') is not False or manifest.get('locked_test_artifact_is_label_blind') is not True:
        raise RuntimeError(f'Stacking seed {seed} melanggar guardrail locked-test blind.')

    checksums = manifest.get('output_checksums', {})
    if set(checksums) != set(R1_07_REQUIRED_OUTPUTS):
        raise RuntimeError(f'Stacking seed {seed}: daftar checksum tidak sama dengan kontrak output.')
    checksum_failures = [
        relative_path for relative_path, expected_hash in checksums.items()
        if sha256_file(seed_dir / relative_path) != expected_hash
    ]
    if checksum_failures:
        raise RuntimeError(f'Stacking seed {seed}: checksum gagal untuk {checksum_failures}')

    source_r1_06_manifest = ROUND1_DIR / 'artifacts' / 'r1_06_repeated_seed_base_oof_v2' / f'seed_{seed}' / 'run_manifest.json'
    if not source_r1_06_manifest.is_file():
        raise FileNotFoundError(f'Base-model source manifest seed {seed} tidak ditemukan.')
    if sha256_file(source_r1_06_manifest) != manifest.get('r1_06_run_manifest_sha256'):
        raise RuntimeError(f'Stacking seed {seed}: base-model source manifest tidak lagi cocok.')

    validation = pd.read_csv(seed_dir / 'validation_model_predictions.csv')
    locked_blind = pd.read_csv(seed_dir / 'locked_test_model_predictions_blind.csv')
    stored_validation_metrics = pd.read_csv(seed_dir / 'validation_metrics.csv')

    assert_exact_row_ids(locked_blind, LOCKED_ROW_IDS, f'Stacking seed {seed}/locked test')
    if len(validation) == 0 or not np.array_equal(validation['row_id'].to_numpy(dtype=np.int64), np.arange(len(validation), dtype=np.int64)):
        raise RuntimeError(f'Stacking seed {seed}: row_id validation tidak berurutan.')
    validate_model_outputs(validation, f'Stacking seed {seed}/validation', require_label=True)
    validate_model_outputs(locked_blind, f'Stacking seed {seed}/locked test blind', require_label=False)

    if len(stored_validation_metrics) != len(MODEL_ORDER):
        raise RuntimeError(f'Stacking seed {seed}: validation_metrics tidak berisi lima model.')
    if set(stored_validation_metrics['model_key']) != {model_key for model_key, _ in MODEL_ORDER}:
        raise RuntimeError(f'Stacking seed {seed}: daftar model validation berubah.')

    recomputed_validation_rows = []
    y_validation = validation['y_silver'].to_numpy(dtype=np.int64)
    for model_key, model_name in MODEL_ORDER:
        result = metric_summary(y_validation, validation[probability_columns(model_key)].to_numpy(dtype=np.float64))
        saved = stored_validation_metrics.loc[stored_validation_metrics['model_key'].eq(model_key)].iloc[0]
        for metric_name in METRIC_COLUMNS:
            if not np.isclose(result[metric_name], float(saved[metric_name]), atol=VALIDATION_METRIC_REPRODUCTION_ATOL, rtol=1e-10):
                raise RuntimeError(f'Stacking seed {seed}/{model_key}: metric validation tidak dapat direproduksi untuk {metric_name}.')
        recomputed_validation_rows.append({
            'seed': seed, 'split': 'validation', 'model_key': model_key, 'model': model_name,
            'probability_source': saved['probability_source'], **result,
        })

    return {
        'seed': seed,
        'seed_dir': seed_dir,
        'manifest': manifest,
        'validation': validation,
        'locked_blind': locked_blind,
        'validation_metric_rows': recomputed_validation_rows,
        'input_hashes': {
            'r1_07_run_manifest_sha256': sha256_file(manifest_path),
            'r1_07_run_state_sha256': sha256_file(state_path),
            'r1_07_locked_blind_prediction_sha256': sha256_file(seed_dir / 'locked_test_model_predictions_blind.csv'),
            'r1_07_validation_prediction_sha256': sha256_file(seed_dir / 'validation_model_predictions.csv'),
        },
    }


### Verify cross-seed comparability

Loads all planned seed outputs, checks that validation and locked-test row identities and recipes agree across seeds, and records only permissible seed-level input metadata.


In [ ]:
seed_inputs = [load_and_validate_r1_07_seed(seed) for seed in PLANNED_SEEDS]

reference_validation_ids = seed_inputs[0]['validation']['row_id'].to_numpy(dtype=np.int64)
reference_validation_labels = seed_inputs[0]['validation']['y_silver'].to_numpy(dtype=np.int64)
for seed_input in seed_inputs[1:]:
    validation = seed_input['validation']
    if not np.array_equal(validation['row_id'].to_numpy(dtype=np.int64), reference_validation_ids):
        raise RuntimeError('Validation row_id tidak konsisten antar-seed.')
    if not np.array_equal(validation['y_silver'].to_numpy(dtype=np.int64), reference_validation_labels):
        raise RuntimeError('Validation silver label tidak konsisten antar-seed.')

protocol_without_seed = []
for seed_input in seed_inputs:
    protocol = dict(seed_input['manifest']['protocol'])
    protocol.pop('seed', None)
    protocol.pop('source_r1_06_protocol_hash', None)
    protocol_without_seed.append(canonical_json_hash(protocol))
if len(set(protocol_without_seed)) != 1:
    raise RuntimeError('Stacking and calibration recipe differs across seeds outside the seed value.')

seed_readiness = pd.DataFrame([
    {
        'seed': item['seed'],
        'r1_07_manifest_sha256': item['input_hashes']['r1_07_run_manifest_sha256'],
        'validation_rows': len(item['validation']),
        'locked_test_blind_rows': len(item['locked_blind']),
        'r1_07_locked_metrics_previously_computed': item['manifest']['locked_test_metrics_computed'],
        'r1_07_locked_artifact_was_label_blind': item['manifest']['locked_test_artifact_is_label_blind'],
    }
    for item in seed_inputs
])
display(seed_readiness)
print('Semua seed lolos integrity check. Tahap berikutnya menghitung metrik terhadap silver labels locked test.')


### Calculate per-seed validation and locked-test metrics

Combines completed seed predictions with validation labels and now-attached locked-test silver labels, then calculates metrics for the five final models without selecting a preferred seed.


In [ ]:
validation_metric_rows = []
locked_metric_rows = []
wide_locked_frames = []
long_locked_frames = []

for seed_input in seed_inputs:
    seed = seed_input['seed']
    validation_metric_rows.extend(seed_input['validation_metric_rows'])
    locked = seed_input['locked_blind'].copy()
    locked.insert(0, 'y_silver', LOCKED_Y_SILVER)
    locked.insert(0, 'seed', seed)
    locked.insert(2, 'y_silver_label', [LABEL_NAMES[index] for index in LOCKED_Y_SILVER])
    wide_locked_frames.append(locked)

    validation_sources = {
        row['model_key']: row['probability_source']
        for row in seed_input['validation_metric_rows']
    }
    for model_key, model_name in MODEL_ORDER:
        probabilities = locked[probability_columns(model_key)].to_numpy(dtype=np.float64)
        metrics = metric_summary(LOCKED_Y_SILVER, probabilities)
        locked_metric_rows.append({
            'seed': seed, 'split': 'locked_test_silver', 'model_key': model_key, 'model': model_name,
            'probability_source': validation_sources[model_key], **metrics,
        })
        long_locked_frames.append(pd.DataFrame({
            'seed': seed,
            'row_id': LOCKED_ROW_IDS,
            'y_silver': LOCKED_Y_SILVER,
            'y_silver_label': [LABEL_NAMES[index] for index in LOCKED_Y_SILVER],
            'model_key': model_key,
            'model': model_name,
            'prob_negatif': probabilities[:, 0],
            'prob_netral': probabilities[:, 1],
            'prob_positif': probabilities[:, 2],
            'prediction': locked[f'{model_key}_prediction'].to_numpy(dtype=np.int64),
            'prediction_label': [LABEL_NAMES[index] for index in locked[f'{model_key}_prediction'].to_numpy(dtype=np.int64)],
            'confidence': locked[f'{model_key}_confidence'].to_numpy(dtype=np.float64),
            'entropy': locked[f'{model_key}_entropy'].to_numpy(dtype=np.float64),
        }))

validation_metrics_per_seed = pd.DataFrame(validation_metric_rows).sort_values(['model_key', 'seed']).reset_index(drop=True)
locked_metrics_per_seed = pd.DataFrame(locked_metric_rows).sort_values(['model_key', 'seed']).reset_index(drop=True)
locked_predictions_by_seed = pd.concat(wide_locked_frames, ignore_index=True)
locked_predictions_per_model_long = pd.concat(long_locked_frames, ignore_index=True)

expected_long_rows = len(PLANNED_SEEDS) * len(LOCKED_ROW_IDS) * len(MODEL_ORDER)
if len(locked_predictions_per_model_long) != expected_long_rows:
    raise RuntimeError('Jumlah row prediction long tidak sesuai kontrak seed x row x model.')

display(locked_metrics_per_seed[['seed', 'model', 'accuracy', 'macro_f1', 'ece', 'brier_score']])


### Summarize variation across seeds

Produces mean, standard deviation, and range summaries for accuracy, macro-F1, ECE, and Brier score across the three completed seeds.


In [ ]:
def summarize_repeated_seed_metrics(metrics_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    group_columns = ['split', 'model_key', 'model', 'probability_source']
    for keys, group in metrics_frame.groupby(group_columns, sort=False):
        record = dict(zip(group_columns, keys))
        record['n_seeds'] = int(group['seed'].nunique())
        record['seeds'] = ','.join(str(seed) for seed in sorted(group['seed'].unique()))
        record['n_per_seed'] = int(group['n'].iloc[0])
        for metric_name in METRIC_COLUMNS:
            values = group[metric_name].to_numpy(dtype=np.float64)
            record[f'{metric_name}_mean'] = float(values.mean())
            record[f'{metric_name}_std'] = float(values.std(ddof=1)) if len(values) > 1 else 0.0
            record[f'{metric_name}_min'] = float(values.min())
            record[f'{metric_name}_max'] = float(values.max())
        rows.append(record)
    return pd.DataFrame(rows).sort_values(['split', 'model_key']).reset_index(drop=True)


all_metrics_per_seed = pd.concat([validation_metrics_per_seed, locked_metrics_per_seed], ignore_index=True)
repeated_seed_metric_summary = summarize_repeated_seed_metrics(all_metrics_per_seed)

stability_rows = []
pairwise_rows = []
for model_key, model_name in MODEL_ORDER:
    model_rows = locked_predictions_per_model_long.loc[locked_predictions_per_model_long['model_key'].eq(model_key)]
    prediction_matrix = model_rows.pivot(index='row_id', columns='seed', values='prediction').reindex(index=LOCKED_ROW_IDS, columns=PLANNED_SEEDS)
    confidence_matrix = model_rows.pivot(index='row_id', columns='seed', values='confidence').reindex(index=LOCKED_ROW_IDS, columns=PLANNED_SEEDS)
    if prediction_matrix.isna().any().any() or confidence_matrix.isna().any().any():
        raise RuntimeError(f'Stabilitas seed {model_key} tidak memiliki seluruh row.')
    unanimous = prediction_matrix.nunique(axis=1).eq(1)
    confidence_std = confidence_matrix.std(axis=1, ddof=1)
    stability_rows.append({
        'model_key': model_key, 'model': model_name, 'n_locked_rows': int(len(prediction_matrix)),
        'unanimous_hard_prediction_n': int(unanimous.sum()),
        'unanimous_hard_prediction_pct': float(unanimous.mean() * 100.0),
        'hard_prediction_varies_n': int((~unanimous).sum()),
        'hard_prediction_varies_pct': float((~unanimous).mean() * 100.0),
        'mean_confidence_std_across_seeds': float(confidence_std.mean()),
        'p95_confidence_std_across_seeds': float(confidence_std.quantile(0.95)),
    })
    for seed_a, seed_b in [(42, 52), (42, 62), (52, 62)]:
        disagreement = prediction_matrix[seed_a].ne(prediction_matrix[seed_b])
        pairwise_rows.append({
            'model_key': model_key, 'model': model_name, 'seed_a': seed_a, 'seed_b': seed_b,
            'n_locked_rows': int(len(disagreement)),
            'hard_prediction_disagreement_n': int(disagreement.sum()),
            'hard_prediction_disagreement_pct': float(disagreement.mean() * 100.0),
        })

locked_seed_stability_summary = pd.DataFrame(stability_rows)
locked_seed_pairwise_disagreement = pd.DataFrame(pairwise_rows)

display(repeated_seed_metric_summary[['split', 'model', 'accuracy_mean', 'accuracy_std', 'macro_f1_mean', 'macro_f1_std', 'ece_mean', 'ece_std', 'brier_score_mean', 'brier_score_std']])
display(locked_seed_stability_summary)


### Write private aggregation artifacts and handoff metadata

Writes the authorized aggregate tables, long prediction files, checksums, and handoff notes for later paired statistics, human-label evaluation, and triage. Row-level files are not public outputs.


In [ ]:
OUTPUT_FILES = [
    'seed_input_readiness.csv',
    'validation_metrics_per_seed.csv',
    'locked_test_silver_metrics_per_seed.csv',
    'repeated_seed_metric_summary.csv',
    'locked_test_predictions_by_seed_with_silver.csv',
    'locked_test_predictions_per_model_long_with_silver.csv',
    'locked_test_seed_prediction_stability_summary.csv',
    'locked_test_seed_pairwise_hard_label_disagreement.csv',
    'prediction_contract.json',
]


def output_is_complete_and_current(protocol_hash: str, input_hashes: dict) -> bool:
    manifest_path = OUTPUT_ROOT / 'run_manifest.json'
    state_path = OUTPUT_ROOT / 'run_state.json'
    if not manifest_path.is_file() or not state_path.is_file():
        return False
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    state = json.loads(state_path.read_text(encoding='utf-8'))
    if state.get('status') != 'completed' or manifest.get('protocol_hash') != protocol_hash:
        return False
    if manifest.get('input_hashes') != input_hashes:
        return False
    checksums = manifest.get('output_checksums', {})
    return set(checksums) == set(OUTPUT_FILES) and all(
        (OUTPUT_ROOT / relative_path).is_file() and sha256_file(OUTPUT_ROOT / relative_path) == expected_hash
        for relative_path, expected_hash in checksums.items()
    )


protocol = {
    'protocol_version': PROTOCOL_VERSION,
    'planned_seeds': PLANNED_SEEDS,
    'model_order': MODEL_ORDER,
    'locked_test_label_source': str(LOCKED_TEST_SOURCE_PATH.relative_to(PROJECT_ROOT)),
    'locked_test_label_column': LABEL_COLUMN,
    'locked_test_evaluation_label': 'LSTM-derived silver labels only',
    'repeated_seed_primary_analysis': 'per-seed metrics with arithmetic mean and sample standard deviation',
    'seed_selection_policy': 'No seed is selected as the winner from locked-test metrics.',
    'multi_seed_probability_ensemble': 'Not created in this protocol.',
    'statistical_testing': 'Not performed here. The paired-statistics stage must not pool repeated locked-test rows as independent observations.',
    'human_label_evaluation': 'Not performed here. The human-label evaluation stage must use completed manual locked-test audit labels separately.',
    'triage_evaluation': 'Not performed here. The triage stage consumes per-seed probabilities from the long prediction artifact.',
}
protocol_hash = canonical_json_hash(protocol)
input_hashes = {
    'locked_test_source_sha256': LOCKED_TEST_SOURCE_SHA256,
    **{f'seed_{item["seed"]}_{key}': value for item in seed_inputs for key, value in item['input_hashes'].items()},
}

if output_is_complete_and_current(protocol_hash, input_hashes):
    print('Repeated-seed evaluation output lengkap dan input tidak berubah. Tidak ada file yang ditulis ulang.')
else:
    started_at = time.time()
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    atomic_write_json(OUTPUT_ROOT / 'run_state.json', {
        'status': 'running', 'started_at_utc': utc_now(), 'protocol_hash': protocol_hash,
    })
    try:
        atomic_write_csv(OUTPUT_ROOT / 'seed_input_readiness.csv', seed_readiness)
        atomic_write_csv(OUTPUT_ROOT / 'validation_metrics_per_seed.csv', validation_metrics_per_seed)
        atomic_write_csv(OUTPUT_ROOT / 'locked_test_silver_metrics_per_seed.csv', locked_metrics_per_seed)
        atomic_write_csv(OUTPUT_ROOT / 'repeated_seed_metric_summary.csv', repeated_seed_metric_summary)
        atomic_write_csv(OUTPUT_ROOT / 'locked_test_predictions_by_seed_with_silver.csv', locked_predictions_by_seed)
        atomic_write_csv(OUTPUT_ROOT / 'locked_test_predictions_per_model_long_with_silver.csv', locked_predictions_per_model_long)
        atomic_write_csv(OUTPUT_ROOT / 'locked_test_seed_prediction_stability_summary.csv', locked_seed_stability_summary)
        atomic_write_csv(OUTPUT_ROOT / 'locked_test_seed_pairwise_hard_label_disagreement.csv', locked_seed_pairwise_disagreement)
        atomic_write_json(OUTPUT_ROOT / 'prediction_contract.json', {
            'seeds': PLANNED_SEEDS,
            'locked_test_rows': int(len(LOCKED_ROW_IDS)),
            'model_order': [{'model_key': key, 'model': name} for key, name in MODEL_ORDER],
            'wide_prediction_file': 'locked_test_predictions_by_seed_with_silver.csv',
            'long_prediction_file': 'locked_test_predictions_per_model_long_with_silver.csv',
            'long_key': ['seed', 'row_id', 'model_key'],
            'silver_label_columns': ['y_silver', 'y_silver_label'],
            'human_audit_join_key': 'row_id',
            'manual_labels_included': False,
            'notes': [
                'Per-seed probabilities and predictions are retained. No best seed is selected.',
                'No probability average across seeds is exported as a new model.',
                'Use the long file for per-seed human-label evaluation, statistics, and triage.',
            ],
        })

        output_checksums = {relative_path: sha256_file(OUTPUT_ROOT / relative_path) for relative_path in OUTPUT_FILES}
        completed_at = utc_now()
        atomic_write_json(OUTPUT_ROOT / 'run_manifest.json', {
            'protocol': protocol, 'protocol_hash': protocol_hash, 'input_hashes': input_hashes,
            'completed_at_utc': completed_at, 'locked_test_metrics_computed': True,
            'evaluation_target': 'LSTM-derived silver labels on the locked test set',
            'output_checksums': output_checksums,
            'row_counts': {
                'locked_test': int(len(LOCKED_ROW_IDS)),
                'seed_count': int(len(PLANNED_SEEDS)),
                'models_per_seed': int(len(MODEL_ORDER)),
                'long_prediction_rows': int(len(locked_predictions_per_model_long)),
            },
            'environment': {
                'python': sys.version, 'platform': platform.platform(),
                'numpy': np.__version__, 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
            },
        })
        atomic_write_json(OUTPUT_ROOT / 'run_state.json', {
            'status': 'completed', 'started_at_utc': datetime.fromtimestamp(started_at, timezone.utc).isoformat(),
            'completed_at_utc': completed_at, 'elapsed_seconds': time.time() - started_at,
            'protocol_hash': protocol_hash,
        })
        print(f'Repeated-seed evaluation completed in {time.time() - started_at:.1f} seconds.')
    except Exception as error:
        atomic_write_json(OUTPUT_ROOT / 'run_state.json', {
            'status': 'failed', 'failed_at_utc': utc_now(), 'protocol_hash': protocol_hash, 'error': repr(error),
        })
        raise


### Perform final aggregation quality checks

Verifies completed state, checksums, metric availability, and long-prediction completeness before declaring the aggregation artifacts ready for the downstream analysis stages.


In [ ]:
manifest_path = OUTPUT_ROOT / 'run_manifest.json'
state_path = OUTPUT_ROOT / 'run_state.json'
if not manifest_path.is_file() or not state_path.is_file():
    raise RuntimeError('Repeated-seed evaluation manifest atau run state belum tersedia.')

r1_08_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
r1_08_state = json.loads(state_path.read_text(encoding='utf-8'))
if r1_08_state.get('status') != 'completed' or r1_08_manifest.get('locked_test_metrics_computed') is not True:
    raise RuntimeError('Repeated-seed evaluation belum selesai atau belum menghitung metrik locked test.')

checksum_failures = [
    relative_path for relative_path, expected_hash in r1_08_manifest['output_checksums'].items()
    if sha256_file(OUTPUT_ROOT / relative_path) != expected_hash
]
if checksum_failures:
    raise RuntimeError(f'Repeated-seed evaluation checksum gagal: {checksum_failures}')

saved_long = pd.read_csv(OUTPUT_ROOT / 'locked_test_predictions_per_model_long_with_silver.csv')
saved_summary = pd.read_csv(OUTPUT_ROOT / 'repeated_seed_metric_summary.csv')
if len(saved_long) != len(PLANNED_SEEDS) * len(LOCKED_ROW_IDS) * len(MODEL_ORDER):
    raise RuntimeError('Long prediction artifact tidak lengkap.')
if set(saved_long['seed']) != set(PLANNED_SEEDS) or set(saved_long['model_key']) != {key for key, _ in MODEL_ORDER}:
    raise RuntimeError('Seed atau daftar model pada long prediction artifact berubah.')

quality_check = pd.DataFrame([{
    'status': r1_08_state['status'],
    'seeds': ','.join(str(seed) for seed in PLANNED_SEEDS),
    'locked_test_rows': len(LOCKED_ROW_IDS),
    'long_prediction_rows': len(saved_long),
    'metric_summary_rows': len(saved_summary),
    'checksums_checked': len(r1_08_manifest['output_checksums']),
    'checksum_failures': len(checksum_failures),
    'manual_labels_used': False,
    'statistical_tests_run': False,
    'triage_run': False,
}])
display(quality_check)
print('Repeated-seed evaluation siap menjadi input human-label evaluation, paired statistical tests, dan triage.')
